$f(x)$ - simple 1D function

$N$ - number of binary bits

$[x_{min}, x_{max}]$ the interval on which the function is defined. Denoted as edges (list of two elements) in code. 

for example let's choose a simple 1D function $f(x) = \sin(\pi x)$ and calculate it's binary indexed tensor.

In [ ]:
import numpy as np
def f(x): 
    return np.sin(np.pi  * x)

def F_tensor1D(fun, N, edges):
    # edges [x_min, x_max]
    M = 2 ** N # num. of grid points
    n = np.arange(M)
    h = (edges[1] - edges[0]) / M # grid spacing
    x = edges[0] + (0.5 + n) * h # x_n
    y = fun(x)

    # ordering where first index changing fastest, and the last index changing slowest
    F = y.reshape((2,)*N, order="F") 
    return x, y, F

Next functions build tensor train by repeatedly appying SVD decomposition.

Rank $r_{k}$ equals 1 when $k \in \{0, N\}$, for $2 \geq k \geq N-1$ the rank is choosen so that the discarded singular values $d_{k,j}$ satisfy $\sqrt{\sum_{j=R_{k}+1} d_{k,j}^{2}} \leq \epsilon$. Where $\epsilon$ is tolerance. 

eps - $\epsilon$ prescribed tolerance $\sqrt(\sum_{j > r_k} d_{k,j}^2) \leq \epsilon$

F_tesnor - $F_{\sigma_{1}, \dots, \sigma_{N}}$ tensor of shape (2, 2, ..., 2)

R_max - maximal rank $r_{k}$ for all $k$

In [ ]:
#decide rank from tolerance
import numpy as np
from numba import njit

@njit(cache=True)
def rank_from_tolerance(SingularValues, eps):
    m = SingularValues.shape[0] # rank of the diagonal matrix of engel values
    EngenValuesSum = 0.0
    CurrentSum = 0.0
    for j in range(m):
        EngenValuesSum += SingularValues[j] * SingularValues[j]
    for r in range(m):
        CurrentSum += SingularValues[r] * SingularValues[r]
        discarded = EngenValuesSum - CurrentSum
        if discarded < 0.0:
            discarded = 0.0
        if np.sqrt(discarded) <= eps:
            return r+1
    return m


def TensorTrainSVD(F_tensor, eps=0.0, R_max=None):
    # Build a tensor train from full F tensor
    # Returns: list of cores, list of singular values, list of ranks
    dimensions = F_tensor.shape
    N = len(dimensions)
    cores, SingularValues, ranks = [], [], [1]
    R = np.array(F_tensor, copy=True)
    r_previous = 1
    for k in range(N - 1):
        d_k = dimensions[k]
        A = R.reshape((r_previous * d_k, -1), order="C") #last idex changes fastes
        U, D, Vt = np.linalg.svd(A, full_matrices=False) 
        SingularValues.append(D.copy())
        if eps == 0.0:
            threshold = np.finfo(D.dtype).eps * max(A.shape) * D[0] # machine precision
            r_new = int(np.sum(D > threshold))  
            r_new = max(r_new, 1) # make sure rank at leat 1
        else:
            r_new = rank_from_tolerance(D, eps)
        if R_max is not None:
            r_new = min(r_new, R_max)
        U = U[:, :r_new]
        D = D[:r_new]
        Vt = Vt[:r_new, :]
        G_new = U.reshape((r_previous, d_k, r_new), order="C")
        cores.append(G_new)
        R = D[:, None] * Vt
        R = R.reshape((r_new,) + dimensions[k + 1:], order="C")
        ranks.append(r_new)
        r_previous = r_new
    G_end = R.reshape((r_previous, dimensions[-1], 1), order="C")
    cores.append(G_end)
    ranks.append(1)
    return cores, SingularValues, ranks


In [ ]:
def N_to_sigmas(n, N):
    sigmas = []
    for i in range(N):
        sigmas.append((n // (2**(i-1))) % 2 )
    return sigmas
